# P7 – Indicateur de capacité d'achat (taux d'effort)
**Contributeur :** Théophyle · **Date :** 09/11/2026

**Pourquoi :** le taux des crédits est le même pour toute la France : seul, il ne peut pas dire quel territoire va baisser plus qu'un autre. La mensualité d'un achat type, rapportée au revenu local, change elle par territoire et par année. C'est la variable de H6 (et une variable « quand » possible pour H7).

**Entrées :** le prix N-1 (table de modélisation), le taux des crédits immobiliers (Banque de France, `taux_credit_mois.parquet`), le niveau de vie médian des communes (Filosofi 2014-2021, `filosofi_commune_2014_2021.parquet`).

**À produire :** pour chaque ligne de la période longue (prix 2017 à 2025), le **taux d'effort** = mensualité d'un prêt à taux fixe pour acheter un bien type au prix N-1, divisée par le revenu mensuel d'un ménage type du territoire. Fichier `fe_capacite_achat_sbv.parquet` (dans `data/`, non suivi par Git), joint par H6 comme l'emploi l'est par H3c.

## 0. Conventions (écrites avant toute mesure, puis figées)

| Convention | Choix | Pourquoi |
|---|---|---|
| Surface de référence | **60 m²** pour un appartement, **100 m²** pour une maison | ordre de grandeur des biens vendus. Pour un type de bien, la surface ne change que l'échelle de l'indicateur, pas le classement des territoires |
| Financement | **100 % du prix emprunté**, sans apport ni frais | formule de la feuille de route. Un apport réduirait tous les taux d'effort dans la même proportion |
| Durée | **25 ans** (300 mensualités) | durée maximale fixée par le HCSF depuis le 1er janvier 2022 |
| Taux | taux des crédits nouveaux à l'habitat de **décembre N-1** (Banque de France) | dernier taux connu au moment de prévoir le prix N |
| Revenu | niveau de vie médian du sous-bassin **× 1,5 unité de consommation**, divisé par 12 | Filosofi donne un niveau de vie par unité de consommation ; 1,5 UC = un couple, ménage type d'un achat |
| Millésime Filosofi | revenus de **N-3** (2021 pour les prix 2024 et 2025) | Filosofi est publié environ deux ans après l'année des revenus : avec N-3, le revenu est connu avant de prévoir N. Les millésimes après 2021 ne sont pas téléchargés |
| Agrégation au sous-bassin | moyenne des médianes communales **pondérée par les ménages fiscaux** | même règle que le feature engineering (section 5) |
| Repère de lecture | **35 %** de taux d'effort | plafond du HCSF pour les crédits immobiliers, contraignant depuis le 1er janvier 2022 |

**Formule de la mensualité (prêt à taux fixe) :** avec K le montant emprunté, t le taux mensuel (taux annuel / 12) et n le nombre de mensualités,

mensualité = K × t / (1 − (1 + t)^(−n))

**Seulement des informations connues en N-1 :** prix N-1, taux de décembre N-1, revenus de N-3.

In [1]:
import numpy as np
import pandas as pd

import commun
from commun import charger_table

# Conventions (section 0) : écrites avant toute mesure, à ne plus changer
ANNEE_DEBUT_LONGUE = 2017       # période longue (P6)
SURFACE_APPARTEMENT = 60        # m²
SURFACE_MAISON = 100            # m²
DUREE_MOIS = 25 * 12            # 25 ans
MOIS_TAUX = 12                  # taux de décembre N-1
UC_MENAGE = 1.5                 # unités de consommation du ménage type (couple)
DECALAGE_FILOSOFI = 3           # revenus de N-3
PLAFOND_HCSF = 35               # % (repère de lecture)
VILLES_ENTIERES = ["75056", "69123", "13055"]   # Paris, Lyon, Marseille : arrondissements gardés

## 1. Taux des crédits : décembre de chaque année
Taux des crédits nouveaux à l'habitat des particuliers (Banque de France), en % par an. Le taux de décembre N-1 sert pour les prix de l'année N.

In [2]:
taux_mois = pd.read_parquet(commun.DATA_OUT / "taux_credit_mois.parquet")
taux = taux_mois[taux_mois["mois"] == MOIS_TAUX][["annee", "taux"]]
taux = taux.rename(columns={"annee": "annee_taux", "taux": "taux_credit"})

# Un seul taux par année, et toutes les années utiles présentes (2016 à 2024)
assert not taux["annee_taux"].duplicated().any(), "Plusieurs taux de décembre pour une même année"
manquantes = sorted(set(range(ANNEE_DEBUT_LONGUE - 1, 2025)) - set(taux["annee_taux"]))
assert not manquantes, "Taux de décembre manquant pour : " + str(manquantes)
taux[(taux["annee_taux"] >= ANNEE_DEBUT_LONGUE - 1) & (taux["annee_taux"] <= 2024)].sort_values("annee_taux")

,annee_taux,taux_credit
216,2016,1.50
228,2017,1.61
240,2018,1.49
252,2019,1.17
264,2020,1.24
276,2021,1.10
288,2022,2.05
300,2023,3.63
312,2024,3.15


## 2. Revenu : niveau de vie médian par sous-bassin et par millésime
Filosofi donne le niveau de vie médian de chaque commune (en euros par an et par unité de consommation). Rattachement au sous-bassin par la table de liaison (codes communes depuis 2003 : les anciens millésimes se rattachent aussi), puis moyenne pondérée par les ménages fiscaux, sur les seules communes dont la médiane et le nombre de ménages sont connus.

In [3]:
filo = pd.read_parquet(commun.DATA_OUT / "filosofi_commune_2014_2021.parquet",
                       columns=["code_commune", "annee", "niveau_vie_median", "NBMEN"])
filo = filo[~filo["code_commune"].isin(VILLES_ENTIERES)]
liaison = pd.read_parquet(commun.DATA_OUT / "liaison_bv_scindes.parquet", columns=["CODGEO_INI", "sous_bassin_de_vie"])

com = filo.merge(liaison, left_on="code_commune", right_on="CODGEO_INI", how="inner")
com = com[com["niveau_vie_median"].notna() & com["NBMEN"].notna()].copy()
com["masse_niveau_vie"] = com["niveau_vie_median"] * com["NBMEN"]

revenu = com.groupby(["sous_bassin_de_vie", "annee"])[["masse_niveau_vie", "NBMEN"]].sum().reset_index()
revenu["niveau_vie_median"] = revenu["masse_niveau_vie"] / revenu["NBMEN"]
revenu = revenu.rename(columns={"annee": "millesime_filosofi"})[["sous_bassin_de_vie", "millesime_filosofi", "niveau_vie_median"]]

# Contrôle : nombre de sous-bassins et niveau de vie médian national, par millésime
revenu.groupby("millesime_filosofi")["niveau_vie_median"].agg(["count", "median"]).round(0)

,count,median
millesime_filosofi,,
2014,1915,19714.0
2015,1915,19856.0
2016,1915,20085.0
2017,1915,20410.0
2018,1915,21040.0
2019,1915,21216.0
2020,1915,21715.0
2021,1915,22366.0


**Comment lire :** le nombre de sous-bassins qui ont un revenu, et la médiane de leur niveau de vie, pour chaque millésime. Le niveau de vie médian national doit monter doucement (environ +1 à +2 % par an, en euros courants), et le nombre de sous-bassins rester proche d'un millésime à l'autre.

## 3. Calcul du taux d'effort
Pour chaque ligne de la période longue (sous-bassin × année × type de bien) : prix N-1 × surface → montant emprunté → mensualité au taux de décembre N-1 sur 25 ans → divisée par le revenu mensuel du ménage type (revenus de N-3).

In [4]:
brute = charger_table(masquer=False)       # toutes les lignes : l'indicateur ne dépend pas du masque
idx = brute.index
lignes = pd.DataFrame({"annee": idx.get_level_values("annee"),
                       "sous_bassin_de_vie": idx.get_level_values("sous_bassin_de_vie"),
                       "type_bien": idx.get_level_values("type_bien"),
                       "prix_n1": brute["prix_n1"].to_numpy()})
lignes = lignes[lignes["annee"] >= ANNEE_DEBUT_LONGUE].copy()

# Correspondance des années : taux de décembre N-1, revenus de N-3 (2021 au plus : derniers téléchargés)
lignes["annee_taux"] = lignes["annee"] - 1
lignes["millesime_filosofi"] = (lignes["annee"] - DECALAGE_FILOSOFI).clip(upper=revenu["millesime_filosofi"].max())
lignes = lignes.merge(taux, on="annee_taux", how="left", validate="many_to_one")
lignes = lignes.merge(revenu, on=["sous_bassin_de_vie", "millesime_filosofi"], how="left", validate="many_to_one")

# Montant emprunté : prix N-1 x surface de référence du type de bien
lignes["surface_m2"] = np.where(lignes["type_bien"] == "maison", SURFACE_MAISON, SURFACE_APPARTEMENT)
lignes["montant_emprunte"] = lignes["prix_n1"] * lignes["surface_m2"]

# Mensualité d'un prêt à taux fixe : K x t / (1 - (1 + t)^(-n)), t = taux mensuel
t = lignes["taux_credit"] / 100 / 12
lignes["mensualite"] = lignes["montant_emprunte"] * t / (1 - (1 + t) ** (-DUREE_MOIS))

# Revenu mensuel du ménage type et taux d'effort (en %)
lignes["revenu_mensuel"] = lignes["niveau_vie_median"] * UC_MENAGE / 12
lignes["taux_effort"] = lignes["mensualite"] / lignes["revenu_mensuel"] * 100

# Contrôles : une ligne par sous-bassin x année x type ; seulement des informations connues avant N
assert not lignes.duplicated(["annee", "sous_bassin_de_vie", "type_bien"]).any()
assert (lignes["annee_taux"] == lignes["annee"] - 1).all()
assert (lignes["millesime_filosofi"] <= lignes["annee"] - DECALAGE_FILOSOFI).all()
print(len(lignes), "lignes (prix 2017 à 2025)")

23498 lignes (prix 2017 à 2025)


## 4. Contrôles
### 4.1 Couverture
L'indicateur existe dès que le prix N-1 et le revenu sont connus. Part des lignes sans taux d'effort parmi celles qui ont un prix N-1, par année et par type de bien, et cause (revenu manquant).

In [5]:
avec_prix_n1 = lignes[lignes["prix_n1"].notna()]
couverture = avec_prix_n1.groupby(["annee", "type_bien"]).agg(
    lignes=("taux_effort", "size"),
    sans_taux_effort_pct=("taux_effort", lambda s: s.isna().mean() * 100),
    sans_revenu_pct=("niveau_vie_median", lambda s: s.isna().mean() * 100))
couverture.round(1).unstack()

lignes        sans_taux_effort_pct        sans_revenu_pct  \
type_bien appartement maison          appartement maison     appartement   
annee                                                                      
2017              746   1776                  0.0    0.0             0.0   
2018              774   1780                  0.0    0.0             0.0   
2019              805   1780                  0.0    0.0             0.0   
2020              857   1780                  0.0    0.0             0.0   
2021              878   1781                  0.0    0.0             0.0   
2022              912   1782                  0.0    0.0             0.0   
2023              861   1781                  0.0    0.0             0.0   
2024              823   1777                  0.0    0.0             0.0   
2025              827   1778                  0.0    0.0             0.0   

                  
type_bien maison  
annee             
2017         0.0  
2018         0.0  
2019         0.0  
2020         0.0  
2021         0.0  
2022         0.0  
2023         0.0  
2024         0.0  
2025         0.0

### 4.2 Ordres de grandeur
Médiane, 5e et 95e centiles du taux d'effort, et part des lignes au-dessus du plafond du HCSF (35 %), par année et par type de bien. Un taux d'effort autour de 30 % est plausible ; 300 % ne l'est pas.

In [6]:
ordres = avec_prix_n1.groupby(["type_bien", "annee"])["taux_effort"].agg(
    p5=lambda s: s.quantile(0.05), mediane="median", p95=lambda s: s.quantile(0.95), maximum="max",
    au_dessus_35_pct=lambda s: (s > PLAFOND_HCSF).mean() * 100)
ordres.round(1)

p5  mediane   p95  maximum  au_dessus_35_pct
type_bien   annee                                                
appartement 2017    8.9     17.8  37.1     68.4               6.0
            2018    9.4     17.9  38.4     73.6               7.0
            2019    8.8     17.2  38.0     76.7               6.2
            2020    8.6     16.5  35.9     76.3               5.4
            2021    8.6     16.8  37.0     81.7               6.5
            2022    8.8     17.3  38.9     79.5               7.6
            2023   10.0     21.1  46.8     84.6              13.5
            2024   12.3     25.0  57.6    107.9              23.2
            2025   11.6     23.7  54.4    104.3              18.7
maison      2017   14.2     23.9  47.0    137.3              16.3
            2018   14.2     24.5  48.7    168.3              18.1
            2019   13.8     24.4  49.5    190.8              18.3
            2020   13.1     23.6  48.2    137.4              16.6
            2021   13.2     24.1  49.2    135.3              18.7
            2022   13.6     25.4  53.2    167.1              22.1
            2023   16.1     30.2  63.5    209.1              36.8
            2024   18.8     35.1  73.8    261.1              50.3
            2025   17.6     32.6  67.9    231.4              43.3

### 4.3 Le taux d'effort suit-il la remontée des taux ?
Taux d'effort médian national par année, à côté du taux de décembre N-1. Avec des taux passés d'environ 1,1 % fin 2021 à plus de 3,5 % fin 2023, le taux d'effort doit monter nettement pour les prix 2023 et 2024.

In [7]:
serie = avec_prix_n1.groupby(["annee", "type_bien"])["taux_effort"].median().unstack().round(1)
serie["taux de décembre N-1 (%)"] = avec_prix_n1.groupby("annee")["taux_credit"].first()
serie

type_bien,appartement,maison,taux de décembre N-1 (%)
annee,,,
2017,17.8,23.9,1.50
2018,17.9,24.5,1.61
2019,17.2,24.4,1.49
2020,16.5,23.6,1.17
2021,16.8,24.1,1.24
2022,17.3,25.4,1.10
2023,21.1,30.2,2.05
2024,25.0,35.1,3.63
2025,23.7,32.6,3.15


### 4.4 Vérification à la main sur deux territoires
On refait le calcul pas à pas, avec les nombres affichés, pour une ligne d'un sous-bassin cher et d'un sous-bassin bon marché (maisons, prix 2024), et on vérifie qu'on retrouve la valeur du tableau.

In [8]:
maisons_2024 = avec_prix_n1[(avec_prix_n1["annee"] == 2024) & (avec_prix_n1["type_bien"] == "maison")
                            & avec_prix_n1["taux_effort"].notna()].sort_values("prix_n1")
exemples = [maisons_2024.iloc[len(maisons_2024) // 20], maisons_2024.iloc[-len(maisons_2024) // 20]]   # environ 5e et 95e centiles du prix

for ligne in exemples:
    k = ligne["prix_n1"] * SURFACE_MAISON
    t_mois = ligne["taux_credit"] / 100 / 12
    m = k * t_mois / (1 - (1 + t_mois) ** (-DUREE_MOIS))
    r = ligne["niveau_vie_median"] * UC_MENAGE / 12
    print("Sous-bassin", ligne["sous_bassin_de_vie"], "| prix 2023 :", round(ligne["prix_n1"]), "€/m²",
          "| emprunt :", round(k), "€ | taux déc. 2023 :", ligne["taux_credit"], "%",
          "| mensualité :", round(m), "€ | revenu mensuel (", ligne["millesime_filosofi"], ") :", round(r), "€",
          "| taux d'effort :", round(m / r * 100, 1), "%")
    assert abs(m / r * 100 - ligne["taux_effort"]) < 1e-9

Sous-bassin 55427 | prix 2023 : 966 €/m² | emprunt : 96600 € | taux déc. 2023 : 3.63 % | mensualité : 490 € | revenu mensuel ( 2021 ) : 2664 € | taux d'effort : 18.4 %
Sous-bassin 56240 | prix 2023 : 4646 €/m² | emprunt : 464600 € | taux déc. 2023 : 3.63 % | mensualité : 2358 € | revenu mensuel ( 2021 ) : 3365 € | taux d'effort : 70.1 %


## 5. Export
Une ligne par sous-bassin × année × type de bien (prix 2017 à 2025). Le fichier va dans `data/processed/`, qui n'est pas suivi par Git ; il se reconstruit en relançant ce notebook.

In [9]:
COLONNES = ["annee", "sous_bassin_de_vie", "type_bien", "taux_effort", "mensualite", "revenu_mensuel",
            "taux_credit", "millesime_filosofi"]
lignes[COLONNES].to_parquet(commun.DATA_OUT / "fe_capacite_achat_sbv.parquet", index=False)
print("Exporté : fe_capacite_achat_sbv.parquet", lignes[COLONNES].shape)

Exporté : fe_capacite_achat_sbv.parquet (23498, 8)


## 6. Utilisation dans un notebook du tour 2 (H6, H7)
La table de modélisation garde son index (annee, sous_bassin_de_vie, type_bien) ; la fonction ajoute la colonne `taux_effort` sans toucher aux autres.

In [10]:
def ajouter_capacite_achat(table):
    """Ajoute le taux d'effort (P7) à la table de modélisation, sur l'année, le sous-bassin et le type de bien."""
    capacite = (pd.read_parquet(commun.DATA_OUT / "fe_capacite_achat_sbv.parquet")
                .set_index(["annee", "sous_bassin_de_vie", "type_bien"])["taux_effort"])
    t = table.copy()
    t["taux_effort"] = capacite.reindex(t.index).to_numpy()
    return t


# Exemple : période longue avec le taux d'effort
essai = ajouter_capacite_achat(charger_table())
essai = essai[essai["annee"] >= ANNEE_DEBUT_LONGUE]
print("Lignes de la période longue :", len(essai), "| sans taux d'effort :", int(essai["taux_effort"].isna().sum()))

Lignes de la période longue : 23489 | sans taux d'effort : 0


## Conclusion
- **Couverture** : le taux d'effort est calculé pour toutes les lignes qui ont un prix N-1, de 2017 à 2025 (23 498 lignes) ; sur la période longue après le masque commun, aucune ligne n'en manque (0 sur 23 489).
- **Seulement des informations connues avant N** : prix N-1, taux de décembre N-1 et revenus de N-3 (contrôles de la section 3).
- **Ordres de grandeur plausibles** : taux d'effort médian de 17 % (appartements) et 24 % (maisons) avant 2023, de 25 % et 35 % pour les prix 2024. Les valeurs extrêmes (jusqu'à 261 % pour les maisons en 2024) concernent les territoires les plus chers ; le 95e centile reste sous 74 %.
- **Il suit la remontée des taux** : pour les maisons, le taux d'effort médian passe de 25,4 % (prix 2022) à 30,2 % (2023) puis 35,1 % (2024), quand le taux de décembre N-1 passe de 1,1 % à 2,05 % puis 3,63 % ; il redescend à 32,6 % en 2025 (taux de 3,15 %). La part des lignes au-dessus du plafond du HCSF (35 %) passe de 22 % à 50 % pour les maisons, de 8 % à 23 % pour les appartements.
- **Vérification à la main** : le calcul pas à pas sur deux sous-bassins (maisons, prix 2024 : 18,4 % et 70,1 %) retrouve exactement les valeurs du tableau.
- **Limites** : un seul ménage type (couple, 1,5 UC) et une surface fixe par type de bien ; prêt sans apport (les niveaux sont donc élevés pour les maisons chères) ; revenus de N-3 (2021 au plus pour 2024 et 2025) ; le taux est national, la variation entre territoires vient du prix et du revenu.
- **Pour H6** : l'indicateur bouge surtout d'une année à l'autre avec les taux (le « quand »), et se différencie entre territoires par le prix et le revenu. L'EDA de H6 dira s'il sépare les territoires en hausse, stables et en baisse.